# Bài 1.1: Chuẩn hóa Bảng mã Unicode, Dấu thanh & Làm sạch Văn bản Tiếng Việt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vietnamese-nlp-mastery/vietnamese-nlp-mastery/blob/main/01_Foundations_Preprocessing/01_unicode_normalization_and_cleaning.ipynb)

> **Mục tiêu bài học:**
> 1. Hiểu sâu sự khác biệt giữa **Unicode Dựng sẵn (`NFC`)** và **Unicode Tổ hợp (`NFD`)** — nguyên nhân số 1 gây lỗi từ điển (Out-of-Vocabulary) âm thầm trong NLP Tiếng Việt.
> 2. Chuẩn hóa vị trí đặt dấu thanh Tiếng Việt (**Kiểu cũ**: `hòa, khỏe, thủy` vs **Kiểu mới**: `hoà, khoẻ, thuỷ`) và hiểu tại sao `PhoBERT` / `VnCoreNLP` nhạy cảm với quy tắc này.
> 3. Xử lý các ký tự điều khiển ẩn (`Zero-Width Space \u200b`, `BOM \ufeff`), lỗi mã hóa console Windows (`cp1252`), và chuẩn hóa Teencode trên dữ liệu thực tế `UIT-VSFC`.

## 1. Bẫy Unicode Tiếng Việt: Dựng sẵn (`NFC`) vs. Tổ hợp (`NFD`)

Trong bảng mã Unicode, một ký tự tiếng Việt có dấu (ví dụ chữ **`ệ`** trong từ **`Việt`**) có thể được biểu diễn theo 2 cách hoàn toàn khác nhau ở mức byte:
- **NFC (Normalization Form C - Dựng sẵn)**: Chữ `ệ` là **1 ký tự duy nhất** (`\u1ec7` - `LATIN SMALL LETTER E WITH CIRCUMFLEX AND DOT BELOW`).
- **NFD (Normalization Form D - Tổ hợp)**: Chữ `ệ` được ghép từ **3 ký tự rời rạc**: chữ `e` (`\u0065`) + dấu nặng (`\u0323`) + dấu mũ (`\u0302`).

**Hậu quả nếu không chuẩn hóa về `NFC`:**
- Hai chuỗi `"Tiếng Việt"` nhìn trên màn hình giống hệt nhau 100%, nhưng phép so sánh `str_nfc == str_nfd` trả về `False`!
- Khi đưa chuỗi `NFD` vào Tokenizer của `PhoBERT`, `ViT5` hoặc `BGE-M3`, từ điển của mô hình (vốn được xây dựng trên `NFC`) sẽ không nhận diện được từ và bẻ vụn thành các subword rác hoặc `[UNK]`.

In [1]:
import unicodedata

# Khởi tạo cùng một từ "Tiếng Việt" ở hai dạng NFC và NFD
text_nfc = unicodedata.normalize("NFC", "Tiếng Việt")
text_nfd = unicodedata.normalize("NFD", "Tiếng Việt")

print(f"Hiển thị NFC : '{text_nfc}'")
print(f"Hiển thị NFD : '{text_nfd}'")
print(f"So sánh bằng (text_nfc == text_nfd): {text_nfc == text_nfd}")
print(f"Độ dài chuỗi len(NFC) = {len(text_nfc)} ký tự | len(NFD) = {len(text_nfd)} ký tự")
print("-" * 65)
print("Phân rã từng codepoint của từ 'Việt' ở dạng NFC vs NFD:")
print("  NFC:", [f"{ch} (U+{ord(ch):04X} - {unicodedata.name(ch)})" for ch in text_nfc.split()[1]])
print("  NFD:", [f"{ch} (U+{ord(ch):04X} - {unicodedata.name(ch)})" for ch in text_nfd.split()[1]])

Hiển thị NFC : 'Tiếng Việt'
Hiển thị NFD : 'Tiếng Việt'
So sánh bằng (text_nfc == text_nfd): False
Độ dài chuỗi len(NFC) = 10 ký tự | len(NFD) = 15 ký tự
-----------------------------------------------------------------
Phân rã từng codepoint của từ 'Việt' ở dạng NFC vs NFD:
  NFC: ['V (U+0056 - LATIN CAPITAL LETTER V)', 'i (U+0069 - LATIN SMALL LETTER I)', 'ệ (U+1EC7 - LATIN SMALL LETTER E WITH CIRCUMFLEX AND DOT BELOW)', 't (U+0074 - LATIN SMALL LETTER T)']
  NFD: ['V (U+0056 - LATIN CAPITAL LETTER V)', 'i (U+0069 - LATIN SMALL LETTER I)', 'e (U+0065 - LATIN SMALL LETTER E)', '̣ (U+0323 - COMBINING DOT BELOW)', '̂ (U+0302 - COMBINING CIRCUMFLEX ACCENT)', 't (U+0074 - LATIN SMALL LETTER T)']


## 2. Chuẩn hóa Vị trí Dấu thanh: Kiểu Cũ (`hòa, thủy`) vs. Kiểu Mới (`hoà, thuỷ`)

Trong tiếng Việt, với các âm tiết kết thúc bằng nguyên âm đôi **`oa`, `oe`, `uy`** (không có phụ âm cuối), tồn tại hai quy tắc đặt dấu thanh:

| Quy tắc | Nguyên tắc | Ví dụ | Tương thích mô hình NLP |
| :--- | :--- | :--- | :--- |
| **Kiểu Cũ (Truyền thống)** | Đặt dấu ở nguyên âm đầu (`o`, `u`) | `hòa bình`, `khỏe mạnh`, `thủy lợi`, `túy lúy` | **Chuẩn của VnCoreNLP, PhoW2V, PhoBERT** |
| **Kiểu Mới (Bộ GD&ĐT)** | Đặt dấu ở nguyên âm chính (`a`, `e`, `y`) | `hoà bình`, `khoẻ mạnh`, `thuỷ lợi`, `tuý luý` | Thường gặp trên báo điện tử hiện đại & bộ gõ macOS |

> **Lưu ý quan trọng**: Riêng tổ hợp **`qu`** (như `quý giá`, `ma quỷ`, `quỳ lạy`), chữ `u` thuộc âm đầu `qu`, nên dấu thanh **luôn luôn đặt trên chữ `y`** ở cả hai kiểu!

In [2]:
import sys
from pathlib import Path

# Import module chuẩn hóa tiếng Việt trong cùng thư mục
sys.path.insert(0, str(Path.cwd()))
from vn_preprocessing import (
    clean_vietnamese_text,
    normalize_teencode,
    normalize_unicode,
    normalize_vietnamese_tone,
)

sentences = [
    "Hoà bình, sức khoẻ, thuỷ lợi và những điều quý giá.",
    "Bộ thuỷ sản khuyến khích nuôi trồng thuỷ hải sản an toàn.",
]

for s in sentences:
    old_style = normalize_vietnamese_tone(s, style="old")
    new_style = normalize_vietnamese_tone(old_style, style="new")
    print(f"Gốc      : {s}")
    print(f"Kiểu Cũ  : {old_style}  <-- Dùng cho PhoBERT / VnCoreNLP")
    print(f"Kiểu Mới : {new_style}")
    print("-" * 65)

Gốc      : Hoà bình, sức khoẻ, thuỷ lợi và những điều quý giá.
Kiểu Cũ  : Hòa bình, sức khỏe, thủy lợi và những điều quý giá.  <-- Dùng cho PhoBERT / VnCoreNLP
Kiểu Mới : Hoà bình, sức khoẻ, thuỷ lợi và những điều quý giá.
-----------------------------------------------------------------
Gốc      : Bộ thuỷ sản khuyến khích nuôi trồng thuỷ hải sản an toàn.
Kiểu Cũ  : Bộ thủy sản khuyến khích nuôi trồng thủy hải sản an toàn.  <-- Dùng cho PhoBERT / VnCoreNLP
Kiểu Mới : Bộ thuỷ sản khuyến khích nuôi trồng thuỷ hải sản an toàn.
-----------------------------------------------------------------


## 3. Làm sạch Ký tự ẩn, Teencode và Áp dụng trên Tập dữ liệu `UIT-VSFC`

Khi thu thập bình luận từ mạng xã hội hoặc khảo sát sinh viên (`UIT-VSFC`), dữ liệu thường chứa:
1. Ký tự không độ rộng (`\u200b` Zero-width space, `\ufeff` BOM) do copy-paste từ web.
2. Từ viết tắt / Teencode (`gv` -> `giảng viên`, `sv` -> `sinh viên`, `ko` -> `không`, `đc` -> `được`).
3. Trộn lẫn Unicode tổ hợp (`NFD`) và dấu thanh kiểu mới.

In [3]:
import json

raw_noisy_samples = [
    "GV​ dạy rất nhiệt tình, ko hiểu chỗ nào đều đc giải đáp vs thái độ hoà nhã.",
    unicodedata.normalize("NFD", "SV ngành CNTT rất thích phần TH xử lý ngôn ngữ tự nhiên."),
]

print("=== KẾT QUẢ LÀM SẠCH VĂN BẢN THỰC TẾ ===")
for idx, raw in enumerate(raw_noisy_samples, 1):
    cleaned = clean_vietnamese_text(raw, tone_style="old", expand_teencode=True)
    print(f"[{idx}] Raw (len={len(raw):02d})    : {raw}")
    print(f"    Cleaned (len={len(cleaned):02d}): {cleaned}\n")

# Đọc thử tập dữ liệu mẫu UIT-VSFC đã khởi tạo trong datasets/samples/
sample_path = Path("../datasets/samples/uit_vsfc_sample.jsonl")
if not sample_path.exists():
    sample_path = Path("datasets/samples/uit_vsfc_sample.jsonl")

if sample_path.exists():
    with open(sample_path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]
    print(f"Đã tải {len(records)} mẫu từ {sample_path.name}. Mẫu đầu tiên:")
    print(json.dumps(records[0], ensure_ascii=False, indent=2))

=== KẾT QUẢ LÀM SẠCH VĂN BẢN THỰC TẾ ===
[1] Raw (len=75)    : GV​ dạy rất nhiệt tình, ko hiểu chỗ nào đều đc giải đáp vs thái độ hoà nhã.
    Cleaned (len=88): giảng viên dạy rất nhiệt tình, không hiểu chỗ nào đều được giải đáp với thái độ hòa nhã.

[2] Raw (len=68)    : SV ngành CNTT rất thích phần TH xử lý ngôn ngữ tự nhiên.
    Cleaned (len=89): sinh viên ngành công nghệ thông tin rất thích phần thực hành xử lý ngôn ngữ tự nhiên.

Đã tải 6 mẫu từ uit_vsfc_sample.jsonl. Mẫu đầu tiên:
{
  "sentence": "Giảng viên dạy rất nhiệt tình, dễ hiểu và luôn giải đáp thắc mắc cho sinh viên.",
  "sentiment": 2,
  "sentiment_label": "Positive",
  "topic": 0,
  "topic_label": "Lecturer"
}
